In [1]:
# ---------- CELL 0: locate the project (same in every notebook) ----------
import os, sys
from pathlib import Path

def find_root(marker="PROJECT_ROOT.marker"):
    here = Path.cwd().resolve()                      # start where this notebook runs
    for folder in [here, *here.parents]:             # go UP one level at a time
        if (folder / marker).exists():
            return folder
    if "google.colab" in sys.modules:                # Colab: look inside Google Drive
        from google.colab import drive
        drive.mount("/content/drive")
        for pattern in (marker, f"*/{marker}", f"*/*/{marker}", f"*/*/*/{marker}"):
            hits = sorted(Path("/content/drive/MyDrive").glob(pattern))
            if hits:
                return hits[0].parent
    raise FileNotFoundError("PROJECT_ROOT.marker not found - finish Day 1 first.")

ROOT = find_root()
os.chdir(ROOT / "notebooks")
RAW, CLEAN = ROOT/"data"/"raw", ROOT/"data"/"clean"
FEATURES, MODEL_OUT = ROOT/"data"/"features", ROOT/"data"/"model_outputs"
SQL_EXPORTS, BI_READY = ROOT/"data"/"sql_exports", ROOT/"data"/"bi_ready"
MODELS, REPORTS, DOCS = ROOT/"models", ROOT/"reports", ROOT/"docs"
FIGURES = REPORTS/"figures"
sys.path.append(str(ROOT/"src"))
print("Project root:", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/Internmo/netflix_churn_project


In [2]:
# ---------- CELL 1: generate the Power Query (M) scripts ----------
import re
import pandas as pd

TABLES_FOR_POWER_BI = [
    "dim_customer", "dim_date", "dim_plan", "dim_content",
    "fact_payments", "fact_viewing", "fact_tickets", "fact_feedback",
    "kpi_summary",
    "model_segment_profile", "model_lift_by_decile", "model_campaign_scenarios", "model_text_theme_summary",
    "sql_cohort_retention", "sql_country_churn_summary", "sql_wau_mau",
    "sql_outreach_list", "sql_winback_candidates", "sql_plan_migration",
]
DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def m_type(series):
    """Pick the Power Query type for one column."""
    non_blank = series.dropna().astype(str)
    if len(non_blank) and non_blank.map(lambda v: bool(DATE_PATTERN.match(v))).all():
        return "type date"
    if pd.api.types.is_integer_dtype(series):
        return "Int64.Type"
    if pd.api.types.is_float_dtype(series):
        return "type number"
    return "type text"

def build_script(table):
    df = pd.read_csv(BI_READY / f"{table}.csv")
    type_list = ",\n        ".join(f'{{"{col}", {m_type(df[col])}}}' for col in df.columns)
    return f'''let
    Source = Csv.Document(
        CsvFiles{{[Name = "{table}.csv"]}}[Content],
        [Delimiter = ",", Encoding = 65001, QuoteStyle = QuoteStyle.Csv]),
    PromotedHeaders = Table.PromoteHeaders(Source, [PromoteAllScalars = true]),
    ChangedTypes = Table.TransformColumnTypes(
        PromotedHeaders,
        {{
        {type_list}
        }},
        "en-US")
in
    ChangedTypes
'''

M_FOLDER = ROOT / "powerbi" / "power_query"
M_FOLDER.mkdir(parents=True, exist_ok=True)
for table in TABLES_FOR_POWER_BI:
    (M_FOLDER / f"{table}.m").write_text(build_script(table))

(M_FOLDER / "_README.txt").write_text("""HOW TO USE THESE FILES (Day 14, Step 5)
For each .m file:
 1. In Power BI: Home > Transform data > New Source > Blank Query.
 2. Home > Advanced Editor. Delete everything and paste the whole text of the .m file. Click Done.
 3. Rename the query (right-hand panel) to the file name WITHOUT .m, for example dim_customer.
Create the two helper queries first (DataFolder and CsvFiles), see the Day 14 guide, Step 4.
""")
print(len(TABLES_FOR_POWER_BI), "scripts written to", M_FOLDER.relative_to(ROOT))
print((M_FOLDER / "kpi_summary.m").read_text())

19 scripts written to powerbi/power_query
let
    Source = Csv.Document(
        CsvFiles{[Name = "kpi_summary.csv"]}[Content],
        [Delimiter = ",", Encoding = 65001, QuoteStyle = QuoteStyle.Csv]),
    PromotedHeaders = Table.PromoteHeaders(Source, [PromoteAllScalars = true]),
    ChangedTypes = Table.TransformColumnTypes(
        PromotedHeaders,
        {
        {"as_of_date", type date},
        {"total_customers", Int64.Type},
        {"active_customers", Int64.Type},
        {"churned_customers", Int64.Type},
        {"churn_rate_pct", type number},
        {"monthly_recurring_revenue", type number},
        {"revenue_last_12m_payments", type number},
        {"high_risk_customers", Int64.Type},
        {"value_at_risk_12m", type number},
        {"churn_model_test_auc", type number}
        },
        "en-US")
in
    ChangedTypes



In [3]:
# ---------- CELL 2: DAX measures ----------
measures = """// ======================= CUSTOMERS =======================
Total Customers = DISTINCTCOUNT ( dim_customer[customer_id] )

Active Customers = CALCULATE ( [Total Customers], dim_customer[customer_status] = "Active" )

Churned Customers = CALCULATE ( [Total Customers], dim_customer[customer_status] = "Churned" )

Churn Rate = DIVIDE ( [Churned Customers], [Total Customers] )

New Customers = CALCULATE ( [Total Customers], USERELATIONSHIP ( dim_date[date], dim_customer[registration_date] ) )

Churned in Period = CALCULATE ( [Churned Customers], USERELATIONSHIP ( dim_date[date], dim_customer[churn_date] ) )

// ======================= REVENUE & PAYMENTS =======================
MRR = CALCULATE ( SUM ( dim_customer[billed_monthly_price] ), dim_customer[customer_status] = "Active" )

Revenue = CALCULATE ( SUM ( fact_payments[amount] ), fact_payments[payment_status] = "Success" )

Payments Attempted = COUNTROWS ( fact_payments )

Failed Payments = CALCULATE ( COUNTROWS ( fact_payments ), fact_payments[payment_status] = "Failed" )

Payment Failure Rate = DIVIDE ( [Failed Payments], [Payments Attempted] )

// ======================= ENGAGEMENT =======================
Sessions = COUNTROWS ( fact_viewing )

Watch Hours = DIVIDE ( SUM ( fact_viewing[watch_duration_minutes] ), 60 )

Avg Completion = DIVIDE ( AVERAGE ( fact_viewing[completion_percentage] ), 100 )

Active Viewers = DISTINCTCOUNT ( fact_viewing[customer_id] )

Hours per Viewer = DIVIDE ( [Watch Hours], [Active Viewers] )

// ======================= SUPPORT & FEEDBACK =======================
Tickets = COUNTROWS ( fact_tickets )

Avg CSAT = AVERAGE ( fact_tickets[customer_satisfaction_score] )

Avg Resolution Hours = AVERAGE ( fact_tickets[resolution_time_hours] )

Negative Feedback Share =
DIVIDE (
    CALCULATE ( COUNTROWS ( fact_feedback ), fact_feedback[sentiment_from_rating] = "Negative" ),
    COUNTROWS ( fact_feedback )
)

// ======================= RISK (from the Python model) =======================
High Risk Customers = CALCULATE ( [Total Customers], dim_customer[risk_tier] = "High" )

Avg Churn Probability = AVERAGE ( dim_customer[churn_probability] )

Value at Risk 12m = SUM ( dim_customer[value_at_risk_12m] )
"""
(ROOT / "powerbi" / "dax_measures.dax").write_text(measures)
print("measures written:", len(re.findall(r"(?m)^[A-Z][A-Za-z0-9 %]* =", measures)), "-> powerbi/dax_measures.dax")

measures written: 23 -> powerbi/dax_measures.dax


In [4]:
# ---------- CELL 3: expected numbers for validating the dashboard ----------
dim_customer = pd.read_csv(BI_READY / "dim_customer.csv")
payments = pd.read_csv(BI_READY / "fact_payments.csv")
viewing = pd.read_csv(BI_READY / "fact_viewing.csv")
tickets = pd.read_csv(BI_READY / "fact_tickets.csv")
feedback = pd.read_csv(BI_READY / "fact_feedback.csv")

active = dim_customer["customer_status"] == "Active"
expected = pd.DataFrame([
    ("Total Customers",       dim_customer["customer_id"].nunique()),
    ("Active Customers",      int(active.sum())),
    ("Churned Customers",     int((~active).sum())),
    ("Churn Rate (%)",        round(100 * (~active).mean(), 2)),
    ("MRR",                   round(dim_customer.loc[active, "billed_monthly_price"].sum(), 2)),
    ("Revenue (successful payments)", round(payments.loc[payments["payment_status"] == "Success", "amount"].sum(), 2)),
    ("Failed Payments",       int((payments["payment_status"] == "Failed").sum())),
    ("Payment Failure Rate (%)", round(100 * (payments["payment_status"] == "Failed").mean(), 2)),
    ("Sessions",              len(viewing)),
    ("Watch Hours",           round(viewing["watch_duration_minutes"].sum() / 60, 1)),
    ("Avg Completion (%)",    round(viewing["completion_percentage"].mean(), 2)),
    ("Tickets",               len(tickets)),
    ("Avg CSAT",              round(tickets["customer_satisfaction_score"].mean(), 2)),
    ("Avg Resolution Hours",  round(tickets["resolution_time_hours"].mean(), 1)),
    ("Negative Feedback Share (%)", round(100 * (feedback["sentiment_from_rating"] == "Negative").mean(), 1)),
    ("High Risk Customers",   int((dim_customer["risk_tier"] == "High").sum())),
    ("Value at Risk 12m",     round(dim_customer["value_at_risk_12m"].sum(), 0)),
], columns=["measure", "expected_value"])
expected.to_csv(ROOT / "powerbi" / "expected_numbers.csv", index=False)
print(expected.to_string(index=False))

with open(DOCS / "project_log.md", "a") as f:
    f.write("""## Day 14 - Power BI setup
- Generated powerbi/power_query/*.m (19 scripts), powerbi/dax_measures.dax, powerbi/expected_numbers.csv.
- Power BI reads ONLY data/bi_ready; the folder path lives in one parameter (DataFolder).
- Built pages 1-3 (Executive Overview, Revenue & Payments, Engagement) in powerbi/netflix_churn_dashboard.pbix.

""")

                      measure  expected_value
              Total Customers         8000.00
             Active Customers         6009.00
            Churned Customers         1991.00
               Churn Rate (%)           24.89
                          MRR        81364.91
Revenue (successful payments)      1396387.71
              Failed Payments         7206.00
     Payment Failure Rate (%)            6.51
                     Sessions        66422.00
                  Watch Hours        53662.10
           Avg Completion (%)           70.47
                      Tickets         6120.00
                     Avg CSAT            3.26
         Avg Resolution Hours           18.00
  Negative Feedback Share (%)           28.50
          High Risk Customers         1201.00
            Value at Risk 12m       151335.00
